# **Week 1: Data Quality Assessment**

**Project:** Heavy Supplier, Inventory & Warehouse Analytics <br>
**Author:** Saw Yu Nandar<br>
**Team:** CX-2026-HSWA-19

In [1]:
import pandas as pd
import numpy as np

# Load core relational datasets
branches_df = pd.read_csv('/content/branches.csv')
customers_df = pd.read_csv('/content/customers.csv')
inventory_df = pd.read_csv('/content/inventory_master.csv')
invoices_df = pd.read_csv('/content/invoices.csv')
payments_df = pd.read_csv('/content/payments.csv')
products_df = pd.read_csv('/content/products.csv')
purchase_orders_header_df = pd.read_csv('/content/purchase_orders_header.csv')
purchase_orders_lines_df = pd.read_csv('/content/purchase_orders_lines.csv')
sales_orders_header_df = pd.read_csv('/content/sales_orders_header.csv')
sales_orders_lines_df = pd.read_csv('/content/sales_orders_lines.csv')
stock_ledger_df = pd.read_csv('/content/stock_ledger.csv')
suppliers_df = pd.read_csv('/content/suppliers.csv')

In [2]:
# Dictionary to manage all datasets efficiently
datasets = {
    'branches': branches_df,
    'customers': customers_df,
    'inventory': inventory_df,
    'invoices': invoices_df,
    'payments': payments_df,
    'products': products_df,
    'purchase_headers': purchase_orders_header_df,
    'purchase_lines': purchase_orders_lines_df,
    'sales_headers': sales_orders_header_df,
    'sales_lines': sales_orders_lines_df,
    'stock_ledger': stock_ledger_df,
    'suppliers': suppliers_df
}

### **1. Total Products**

In [3]:
if "product_id" in products_df.columns:
    total_products = products_df["product_id"].nunique()
else:
    total_products = len(products_df)

print("Total Products:", total_products)

Total Products: 30


### **2. Total Customers**

In [4]:
# 4. Total Customers
if "customer_id" in customers_df.columns:
    total_customers = customers_df["customer_id"].nunique()
else:
    total_customers = len(customers_df)

print("Total Customers:", total_customers)

Total Customers: 500


### **3. Total Suppliers**

In [6]:
# 5. Total Suppliers
if "supplier_id" in suppliers_df.columns:
    total_suppliers = suppliers_df["supplier_id"].nunique()
else:
    total_suppliers = len(suppliers_df)

print("Total Suppliers:", total_suppliers)

Total Suppliers: 8


### **4. Total Sales**

In [5]:
print("Sales Header Columns:", sales_orders_header_df.columns.tolist())
print("Sales Lines Columns:", sales_orders_lines_df.columns.tolist())

total_sales = 0.0

# 1. Check if total exists directly in Sales Header
header_sales_cols = [c for c in ['total_amount', 'total_sales', 'order_total', 'amount'] if c in sales_orders_header_df.columns]

if header_sales_cols:
    sales_col = header_sales_cols[0]
    total_sales = sales_orders_header_df[sales_col].sum()
    print(f"\nCalculated from Sales Header using column '{sales_col}'")

# 2. Otherwise calculate from Sales Lines (quantity * unit_price)
elif 'quantity' in sales_orders_lines_df.columns and 'unit_price' in sales_orders_lines_df.columns:
    total_sales = (sales_orders_lines_df['quantity'] * sales_orders_lines_df['unit_price']).sum()
    print("\nCalculated from Sales Lines using 'quantity' * 'unit_price'")

# 3. Fallback check for line item total column in Sales Lines
elif 'line_total' in sales_orders_lines_df.columns or 'total_price' in sales_orders_lines_df.columns:
    line_col = [c for c in ['line_total', 'total_price'] if c in sales_orders_lines_df.columns][0]
    total_sales = sales_orders_lines_df[line_col].sum()
    print(f"\nCalculated from Sales Lines using column '{line_col}'")

print("Total Sales: £{:,.2f}".format(total_sales))

Sales Header Columns: ['so_id', 'customer_id', 'branch_id', 'order_date', 'delivery_date', 'order_status', 'payment_terms', 'total_order_value', 'total_gst_amount', 'grand_total', 'sales_channel']
Sales Lines Columns: ['so_id', 'line_number', 'product_id', 'quantity', 'unit_price', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total']

Calculated from Sales Lines using 'quantity' * 'unit_price'
Total Sales: £25,447,392,560.00


### **5. Total Sales Quality**

In [7]:
qty_cols = [c for c in ['quantity', 'order_quantity', 'qty', 'units_sold'] if c in sales_orders_lines_df.columns]

if qty_cols:
    sales_qty_col = qty_cols[0]
    total_sales_quantity = sales_orders_lines_df[sales_qty_col].sum()
    print(f"Total Sales Quantity (using '{sales_qty_col}'): {total_sales_quantity:,.0f}")
else:
    total_sales_quantity = 0
    print("Warning: Sales quantity column not found.")

Total Sales Quantity (using 'quantity'): 1,368,534


### **6. Total Inventory Quantity**

In [11]:
possible_qty_names = [
    'quantity', 'stock_quantity', 'stock_level', 'qty_on_hand',
    'units', 'current_stock', 'qty', 'quantity_on_hand', 'available_stock'
]

# Find matching column in inventory_df
inventory_cols_lower = {col.lower(): col for col in inventory_df.columns}
matched_qty_col = next((inventory_cols_lower[name] for name in possible_qty_names if name in inventory_cols_lower), None)

if matched_qty_col:
    total_inventory_quantity = inventory_df[matched_qty_col].sum()
    print(f"Total Inventory Quantity (from inventory_df['{matched_qty_col}']): {total_inventory_quantity:,.0f}")

# Fallback: Check stock_ledger_df if inventory_df doesn't hold quantities directly
elif 'stock_ledger_df' in globals() and not stock_ledger_df.empty:
    ledger_cols_lower = {col.lower(): col for col in stock_ledger_df.columns}
    matched_ledger_col = next((ledger_cols_lower[name] for name in possible_qty_names if name in ledger_cols_lower), None)

    if matched_ledger_col:
        total_inventory_quantity = stock_ledger_df[matched_ledger_col].sum()
        print(f"Total Inventory Quantity (from stock_ledger_df['{matched_ledger_col}']): {total_inventory_quantity:,.0f}")
    else:
        total_inventory_quantity = 0
        print("Warning: No inventory quantity column identified in stock_ledger_df.")
else:
    total_inventory_quantity = 0
    print("Warning: Inventory quantity column not found in inventory_df.")
    print("Available Inventory Columns:", inventory_df.columns.tolist())

Total Inventory Quantity (from inventory_df['current_stock']): 19,303,266


### **7. Total Inventory Value**

In [12]:
total_inventory_value = 0.0

# Search for unit cost/price columns across inventory_df and products_df
cost_names = ['unit_cost', 'cost', 'unit_price', 'price', 'purchase_price']

inv_cost_col = next((inventory_cols_lower[name] for name in cost_names if name in inventory_cols_lower), None)

# Case 1: Quantity and Cost both exist inside inventory_df
if matched_qty_col and inv_cost_col:
    total_inventory_value = (inventory_df[matched_qty_col] * inventory_df[inv_cost_col]).sum()
    print(f"Calculated Inventory Value using inventory_df['{matched_qty_col}'] * inventory_df['{inv_cost_col}']")

# Case 2: Quantity in inventory_df, Cost merged from products_df
elif matched_qty_col and 'product_id' in inventory_df.columns and 'product_id' in products_df.columns:
    prod_cols_lower = {col.lower(): col for col in products_df.columns}
    prod_cost_col = next((prod_cols_lower[name] for name in cost_names if name in prod_cols_lower), None)

    if prod_cost_col:
        merged_inv = inventory_df.merge(
            products_df[['product_id', prod_cost_col]],
            on='product_id',
            how='left'
        )
        total_inventory_value = (merged_inv[matched_qty_col] * merged_inv[prod_cost_col]).sum()
        print(f"Calculated Inventory Value by merging products_df['{prod_cost_col}']")

# Case 3: Direct inventory value column exists
elif any(c in inventory_cols_lower for c in ['total_value', 'inventory_value', 'stock_value']):
    val_col = next(inventory_cols_lower[c] for c in ['total_value', 'inventory_value', 'stock_value'] if c in inventory_cols_lower)
    total_inventory_value = inventory_df[val_col].sum()
    print(f"Calculated Inventory Value directly from inventory_df['{val_col}']")

print("Total Inventory Value: £{:,.2f}".format(total_inventory_value))

Calculated Inventory Value by merging products_df['unit_cost']
Total Inventory Value: £281,292,677,410.00


### **8. KPI Summary Table**

In [13]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Products",
        "Total Customers",
        "Total Suppliers",
        "Total Sales",
        "Total Sales Quantity",
        "Total Inventory Quantity",
        "Total Inventory Value"
    ],
    "Value": [
        f"{total_products:,}",
        f"{total_customers:,}",
        f"{total_suppliers:,}",
        f"£{total_sales:,.2f}",
        f"{total_sales_quantity:,}",
        f"{total_inventory_quantity:,}",
        f"£{total_inventory_value:,.2f}"
    ]
})

kpi_summary

,KPI,Value
0,Total Products,30
1,Total Customers,500
2,Total Suppliers,8
3,Total Sales,"£25,447,392,560.00"
4,Total Sales Quantity,"1,368,534"
5,Total Inventory Quantity,"19,303,266"
6,Total Inventory Value,"£281,292,677,410.00"
